# Residea.ai - XGBoost Ranker Training on GPU

This notebook trains an XGBoost ranking model for property recommendations.

## Setup & Installation


In [ ]:
# Install required packages
%pip install -q pandas numpy scikit-learn xgboost joblib


In [ ]:
# Check GPU availability
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print("✅ GPU Available!")
    print(result.stdout)
    USE_GPU = True
else:
    print("⚠️  No GPU detected, will use CPU")
    USE_GPU = False


## Data Upload

Upload your data files:
- `properties_data.csv`
- `user preferences.csv`
- `crime_rate.csv`
- `update_osm.csv`


In [ ]:
# Create data directory structure
import os
os.makedirs("data/cleandataset", exist_ok=True)
os.makedirs("models_artifacts", exist_ok=True)

print("📁 Directory structure created")


In [ ]:
# Upload files using Colab file uploader
from google.colab import files
import shutil

print("📤 Please upload the following files:")
print("1. properties_data.csv")
print("2. user preferences.csv")
print("3. crime_rate.csv")
print("4. update_osm.csv")

# Uncomment to use file uploader
# uploaded = files.upload()
# for filename in uploaded.keys():
#     shutil.move(filename, f"data/{filename}")
#     print(f"✅ Moved {filename} to data/")

# OR mount Google Drive if files are already there
# from google.colab import drive
# drive.mount('/content/drive')
# # Copy files from drive to data directory
# # shutil.copytree('/content/drive/MyDrive/Residea.ai/data', 'data', dirs_exist_ok=True)


## Module 1: Data Cleaning & Preprocessing

This section contains all functions from `backend/preprocessing/cleaning.py`


In [ ]:
# ============================================================================
# CLEANING.PY - Data Cleaning and Feature Engineering
# ============================================================================

from __future__ import annotations
import math
import os
import re
from dataclasses import dataclass
from typing import Dict, Tuple
import numpy as np
import pandas as pd

# Encoders / mappings
TYPE_ENCODER: Dict[str, int] = {"house": 1, "flat": 2, "plot": 3}
CONDITION_SCORE: Dict[str, float] = {
    "brand new": 1.0,
    "excellent": 0.9,
    "good": 0.7,
    "old renovated": 0.5,
    "old": 0.3,
}
SECURITY_MAP = {"Very Important": 1.5, "Important": 1.3, "Medium": 1.2, "Low": 1.0, "Somewhat Important": 1.2}


@dataclass
class DataBundle:
    properties: pd.DataFrame
    preferences: pd.DataFrame
    crime: pd.DataFrame
    pois: pd.DataFrame


def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def load_raw_data(base_dir: str = "data") -> DataBundle:
    props = pd.read_csv(os.path.join(base_dir, "properties_data.csv"))
    prefs = pd.read_csv(os.path.join(base_dir, "user preferences.csv"))
    crime = pd.read_csv(os.path.join(base_dir, "crime_rate.csv"))
    pois = pd.read_csv(os.path.join(base_dir, "update_osm.csv"))
    return DataBundle(props, prefs, crime, pois)


def normalize_location(name: str) -> str:
    if not isinstance(name, str):
        return ""
    return re.sub(r"[^a-zA-Z0-9]+", "", name).lower()


def parse_budget_range(text: str) -> Tuple[float, float]:
    """Convert '45 Crore - 70 Crore' to numeric PKR."""
    if not isinstance(text, str):
        return (np.nan, np.nan)
    parts = re.split(r"-|to", text)
    if len(parts) < 2:
        return (np.nan, np.nan)
    nums = []
    for raw in parts[:2]:
        raw = raw.strip().lower().replace(",", "")
        mult = 1.0
        if "crore" in raw:
            mult = 1e7
        elif "lakh" in raw:
            mult = 1e5
        found = re.findall(r"\d+\.?\d*", raw)
        nums.append(float(found[0]) * mult if found else np.nan)
    return (nums[0], nums[1])


def haversine(point: np.ndarray, facilities: np.ndarray) -> np.ndarray:
    dlat = facilities[:, 0] - point[0]
    dlon = facilities[:, 1] - point[1]
    a = np.sin(dlat / 2) ** 2 + np.cos(point[0]) * np.cos(facilities[:, 0]) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371.0 * c


def compute_nearest_distance_km(points: np.ndarray, facilities: np.ndarray) -> np.ndarray:
    if len(facilities) == 0:
        return np.full(points.shape[0], np.nan)
    points_rad = np.radians(points)
    facilities_rad = np.radians(facilities)
    distances = []
    for pt in points_rad:
        dists = haversine(pt, facilities_rad)
        distances.append(dists.min())
    return np.array(distances)


def extract_facilities(pois: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    """Return POI subsets for hospital/school/metro/park."""
    pois = pois.copy()
    pois["Latitude"] = pd.to_numeric(pois["Latitude"], errors="coerce")
    pois["Longitude"] = pd.to_numeric(pois["Longitude"], errors="coerce")
    pois["Category"] = pois["Category"].str.lower()
    facilities = {
        "hospital": pois[pois["Category"].str.contains("hospital", na=False)],
        "school": pois[pois["Category"].str.contains("school|college|university", na=False)],
        "metro": pois[pois["Category"].str.contains("metro|train|subway", na=False)],
        "park": pois[pois["Category"].str.contains("park", na=False)],
    }
    return facilities


def clean_properties(df: pd.DataFrame, crime_df: pd.DataFrame, poi_df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.drop_duplicates(subset=["links"])

    for col in ["clean_price", "clean_area", "bedrooms", "bathrooms", "days_posted", "latitude", "longitude"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["bedrooms"] = df["bedrooms"].fillna(df["bedrooms"].median()).clip(lower=0)
    df["bathrooms"] = df["bathrooms"].fillna(df["bathrooms"].median()).clip(lower=0)
    df["clean_area"] = df["clean_area"].replace(0, np.nan)
    df["clean_area"] = df["clean_area"].fillna(df["clean_area"].median())
    df["clean_price"] = df["clean_price"].fillna(df["clean_price"].median())
    df["days_posted"] = df["days_posted"].fillna(df["days_posted"].median()).clip(upper=90)

    df["price_per_sqft"] = df["clean_price"] / df["clean_area"]
    df["price_per_sqft"] = df["price_per_sqft"].replace([np.inf, -np.inf], np.nan).fillna(
        df["price_per_sqft"].median()
    )

    df["clean_location"] = df["clean_location"].fillna(df["location"])
    df["location_key"] = df["clean_location"].apply(normalize_location)

    df["condition_clean"] = df["Condition"].fillna("unknown").str.title()
    df["condition_score"] = (
        df["condition_clean"].str.lower().map(CONDITION_SCORE).fillna(0.5)
    )
    df["is_brand_new"] = (df["condition_clean"].str.lower() == "brand new").astype(int)

    df["property_type"] = df["type"].fillna("unknown").str.lower()
    df["type_encoded"] = df["property_type"].map(TYPE_ENCODER).fillna(0).astype(int)

    df["amenities_count"] = df["Features"].fillna("").apply(lambda x: len([a for a in x.split(",") if a.strip()]))
    df["luxury_indicator"] = ((df["amenities_count"] >= 5) | (df["clean_area"] >= 5000)).astype(int)

    crime_df = crime_df.dropna(subset=["Name"]).copy()
    crime_df["Adjusted_Index_num"] = pd.to_numeric(crime_df["Adjusted_Index"], errors="coerce")
    crime_df = crime_df.dropna(subset=["Adjusted_Index_num"])
    crime_df["crime_location_key"] = crime_df["Name"].apply(normalize_location)
    df = df.merge(
        crime_df[["crime_location_key", "Adjusted_Index_num"]],
        left_on="location_key",
        right_on="crime_location_key",
        how="left",
    )
    median_crime = crime_df["Adjusted_Index_num"].median()
    df["crime_index"] = df["Adjusted_Index_num"].fillna(median_crime)
    df.drop(columns=["crime_location_key", "Adjusted_Index_num"], inplace=True)

    # Normalize crime to 0-1 for safety scoring
    crime_min, crime_max = df["crime_index"].min(), df["crime_index"].max()
    df["crime_normalized"] = (df["crime_index"] - crime_min) / (crime_max - crime_min + 1e-9)
    df["safety_score"] = 1 - df["crime_normalized"]

    facilities = extract_facilities(poi_df)
    for name, poi_subset in facilities.items():
        coords = poi_subset.dropna(subset=["Latitude", "Longitude"])
        df[f"dist_to_{name}"] = compute_nearest_distance_km(
            df[["latitude", "longitude"]].to_numpy(dtype=float),
            coords[["Latitude", "Longitude"]].to_numpy(dtype=float),
        )
        df[f"{name}_score"] = 1 / (1 + df[f"dist_to_{name}"])

    return df


def derive_preference_features(prefs_df: pd.DataFrame) -> pd.DataFrame:
    prefs = prefs_df.copy()
    min_budget, max_budget = zip(*prefs["Budget Range"].apply(parse_budget_range))
    prefs["budget_min"] = min_budget
    prefs["budget_max"] = max_budget
    prefs["budget_center"] = (prefs["budget_min"] + prefs["budget_max"]) / 2
    prefs["budget_range"] = prefs["budget_max"] - prefs["budget_min"]

    prefs["preferred_type"] = prefs["Preferred Property Type"].str.lower().fillna("")
    prefs["preferred_location"] = prefs["Preferred Location"].fillna("")
    prefs["preferred_location_key"] = prefs["preferred_location"].apply(normalize_location)

    prefs["preferred_bedrooms"] = (
        prefs["Number of Bedrooms"].replace({"6+": 6, "4+": 4}).apply(pd.to_numeric, errors="coerce").fillna(3)
    )
    prefs["preferred_area"] = prefs.get("clean_area_sqft", pd.Series([], dtype=float)).fillna(np.nan)

    prefs["security_importance"] = prefs["Importance of Security"].map(SECURITY_MAP).fillna(1.2)
    prefs["purpose_encoded"] = prefs["Purpose of Investment"].str.lower().map(
        {"living ( self use )": 1, "rental income": 2, "resale ( profit )": 3}
    ).fillna(0)
    return prefs


def build_clean_datasets(base_dir: str = "data", output_dir: str = "data/cleandataset") -> Tuple[pd.DataFrame, pd.DataFrame]:
    ensure_dir(output_dir)
    bundle = load_raw_data(base_dir)
    props = clean_properties(bundle.properties, bundle.crime, bundle.pois)
    prefs = derive_preference_features(bundle.preferences)

    props.to_csv(os.path.join(output_dir, "properties_clean.csv"), index=False)
    prefs.to_csv(os.path.join(output_dir, "preferences_enriched.csv"), index=False)
    return props, prefs


print("✅ Cleaning module loaded")


In [ ]:
# ============================================================================
# MATCHING.PY - User-Property Matching Feature Computation
# ============================================================================

from __future__ import annotations
import math
from typing import Dict, Iterable, List, Tuple
import numpy as np
import pandas as pd

# Import from cleaning module (already defined above)
# These are already available: CONDITION_SCORE, SECURITY_MAP, extract_facilities, haversine, normalize_location, parse_budget_range

FACILITY_THRESHOLDS = {"school": 1.5, "hospital": 3.0, "metro": 2.0, "park": 2.0}


def compute_location_centroids(df: pd.DataFrame) -> Dict[str, Tuple[float, float]]:
    centroids = {}
    grouped = df.groupby("location_key")[["latitude", "longitude"]].mean()
    for key, row in grouped.iterrows():
        centroids[key] = (row.latitude, row.longitude)
    return centroids


def location_distance_km(prop_lat: float, prop_lon: float, target_lat: float, target_lon: float) -> float:
    if any(math.isnan(x) for x in [prop_lat, prop_lon, target_lat, target_lon]):
        return float("inf")
    p = np.radians(np.array([prop_lat, prop_lon]))
    t = np.radians(np.array([[target_lat, target_lon]]))
    return float(haversine(p, t).item())


def price_fit_score(clean_price: float, budget_center: float, budget_range: float) -> float:
    if budget_range <= 0 or math.isnan(clean_price) or math.isnan(budget_center):
        return 0.0
    score = 1 - abs(clean_price - budget_center) / (budget_range / 2)
    return float(np.clip(score, 0, 1))


def area_match_score(clean_area: float, preferred_area: float) -> float:
    if preferred_area <= 0 or math.isnan(clean_area):
        return 0.0
    score = 1 - abs(clean_area - preferred_area) / preferred_area
    return float(np.clip(score, 0, 1))


def facility_scores(property_row: pd.Series, facilities: Dict[str, pd.DataFrame]) -> Dict[str, float]:
    scores = {}
    for name, poi_subset in facilities.items():
        coords = poi_subset[["Latitude", "Longitude"]].dropna().to_numpy(dtype=float)
        if len(coords) == 0:
            scores[name] = math.nan
            continue
        pt = np.radians(np.array([property_row["latitude"], property_row["longitude"]]))
        dist = haversine(pt, np.radians(coords)).min()
        scores[name] = dist
    return scores


def compute_match_features(properties: pd.DataFrame, prefs_row: pd.Series, poi_df: pd.DataFrame) -> pd.DataFrame:
    """Return a DataFrame with match features for one user preference row."""
    prefs = prefs_row
    facilities = extract_facilities(poi_df)
    centroids = compute_location_centroids(properties)

    preferred_key = normalize_location(prefs.get("Preferred Location", ""))
    target_lat, target_lon = centroids.get(preferred_key, (math.nan, math.nan))

    def _row_apply(row: pd.Series) -> pd.Series:
        dist_loc = location_distance_km(row["latitude"], row["longitude"], target_lat, target_lon)
        loc_match_score = math.exp(-0.15 * dist_loc) if math.isfinite(dist_loc) else 0.0
        loc_exact = int(row.get("clean_location", "").lower() == prefs.get("Preferred Location", "").lower())

        price_score = price_fit_score(row["clean_price"], prefs["budget_center"], prefs["budget_range"])
        bed_match = int(row["bedrooms"] >= prefs["preferred_bedrooms"])
        area_score = area_match_score(row["clean_area"], prefs.get("preferred_area", math.nan))

        facility_dists = facility_scores(row, facilities)
        facility_scores_inv = {k: 1 / (1 + v) if math.isfinite(v) else 0 for k, v in facility_dists.items()}
        facility_hits = [
            1
            for k, v in facility_dists.items()
            if math.isfinite(v) and v <= FACILITY_THRESHOLDS.get(k, float("inf"))
        ]
        required = [
            "near school" in str(prefs.get("Nearby Facilities", "")).lower(),
            "near hospital" in str(prefs.get("Nearby Facilities", "")).lower(),
            "near metro" in str(prefs.get("Nearby Facilities", "")).lower(),
        ]
        required_count = sum(required) if any(required) else len(FACILITY_THRESHOLDS)
        facility_match_ratio = len(facility_hits) / required_count if required_count else 0

        safety_score = row.get("safety_score", 0.0)
        security_weight = prefs.get("security_importance", 1.2)

        # Simple ROI proxies using location price differentials and crime
        sector_avg = row.get("sector_avg_price_per_sqft", row["price_per_sqft"])
        global_median = properties["price_per_sqft"].median()
        sector_growth = (sector_avg - global_median) / (global_median + 1e-9)
        roi_estimate_1yr = sector_growth * row.get("condition_score", 0.5)
        roi_estimate_5yr = sector_growth * (1 - row.get("crime_normalized", 0.5))
        risk_adjusted_roi = roi_estimate_5yr * safety_score

        return pd.Series(
            {
                "price_fit_score": price_score,
                "bedroom_match": bed_match,
                "area_match_score": area_score,
                "location_exact_match": loc_exact,
                "location_match_distance_km": dist_loc,
                "location_match_score": loc_match_score,
                "dist_to_school": facility_dists.get("school", math.nan),
                "dist_to_hospital": facility_dists.get("hospital", math.nan),
                "dist_to_metro": facility_dists.get("metro", math.nan),
                "dist_to_park": facility_dists.get("park", math.nan),
                "school_score": facility_scores_inv.get("school", 0),
                "hospital_score": facility_scores_inv.get("hospital", 0),
                "metro_score": facility_scores_inv.get("metro", 0),
                "park_score": facility_scores_inv.get("park", 0),
                "facility_match_ratio": facility_match_ratio,
                "safety_score": safety_score,
                "security_weight": security_weight,
                "risk_adjusted_safety": safety_score * security_weight,
                "roi_estimate_1yr": roi_estimate_1yr,
                "roi_estimate_5yr": roi_estimate_5yr,
                "risk_adjusted_roi": risk_adjusted_roi,
            }
        )

    # Pre-compute sector averages to enrich rows
    sector_avg = properties.groupby("clean_location")["price_per_sqft"].median().rename("sector_avg_price_per_sqft")
    props = properties.join(sector_avg, on="clean_location")
    match_features = props.apply(_row_apply, axis=1)
    return pd.concat([props.reset_index(drop=True), match_features.reset_index(drop=True)], axis=1)


print("✅ Matching module loaded")


In [ ]:
# Run data cleaning
print("🧹 Cleaning and preprocessing data...")
props, prefs = build_clean_datasets()
print(f"✅ Clean properties -> data/cleandataset/properties_clean.csv ({len(props)} rows)")
print(f"✅ Enriched preferences -> data/cleandataset/preferences_enriched.csv ({len(prefs)} rows)")


## Module 3: Ranker Training

This section contains all functions from `backend/models/ranker.py`


In [ ]:
# ============================================================================
# RANKER.PY - XGBoost Ranker Training
# ============================================================================

from __future__ import annotations
import os
from dataclasses import dataclass
from typing import List, Tuple
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import ndcg_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from xgboost import XGBRanker

ARTIFACT_DIR = "models_artifacts"
# Set MAX_PREFS to limit number of users for faster testing (None = use all)
MAX_PREFS = None  # Change to 50 or 100 for quick testing


@dataclass
class RankerArtifacts:
    model_path: str
    metrics: dict


def _build_preprocessor(feature_cols: list[str]) -> ColumnTransformer:
    numeric_pipe = Pipeline(steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
    return ColumnTransformer(transformers=[("num", numeric_pipe, feature_cols)], remainder="drop")


def _derive_label(row: pd.Series) -> float:
    # Weighted combination of match signals
    components = [
        row.get("price_fit_score", 0),
        row.get("bedroom_match", 0) * 0.5,
        row.get("area_match_score", 0.5),
        row.get("location_match_score", 1.0),
        row.get("facility_match_ratio", 0.5),
        row.get("risk_adjusted_safety", 0.5),
        row.get("risk_adjusted_roi", 0.2),
    ]
    val = sum(components)
    return float(val)


def _deduplicate_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Remove duplicate column names, keeping the last occurrence."""
    if df.columns.duplicated().any():
        # Reverse columns, deduplicate keeping first (which is last in original), reverse back
        df_reversed = df.iloc[:, ::-1]
        df_dedup = df_reversed.loc[:, ~df_reversed.columns.duplicated()]
        return df_dedup.iloc[:, ::-1]
    return df


def build_matches_for_users(
    props: pd.DataFrame, prefs: pd.DataFrame, poi_df: pd.DataFrame, user_ids: List[int]
) -> Tuple[pd.DataFrame, np.ndarray, list[int]]:
    rows = []
    group_sizes = []
    for i, idx in enumerate(user_ids, start=1):
        pref = prefs.loc[idx]
        matches = compute_match_features(props, pref, poi_df)
        # Deduplicate columns (compute_match_features may create duplicates)
        matches = _deduplicate_columns(matches)
        matches["label"] = matches.apply(_derive_label, axis=1)
        matches["group"] = idx
        rows.append(matches)
        group_sizes.append(len(matches))
        if i % 10 == 0 or i == len(user_ids):
            print(f"   built matches for {i}/{len(user_ids)} users")
    full = pd.concat(rows, axis=0, ignore_index=True)
    # Final deduplication after concat
    full = _deduplicate_columns(full)
    labels = full["label"].to_numpy()
    return full, labels, group_sizes


def mean_ndcg(labels: np.ndarray, preds: np.ndarray, group_sizes: list[int]) -> float:
    scores = []
    start = 0
    for g in group_sizes:
        end = start + g
        y = labels[start:end]
        p = preds[start:end]
        if len(y) > 0:
            scores.append(ndcg_score([y], [p]))
        start = end
    return float(np.mean(scores)) if scores else 0.0


print("✅ Ranker module loaded")


## Step 2: Train XGBoost Ranker

This will train the ranking model using GPU if available.


In [ ]:
def train_ranker(clean_props_path: str, prefs_path: str, use_gpu: bool = True) -> RankerArtifacts:
    print("📌 Loading cleaned datasets...")
    ensure_dir(ARTIFACT_DIR)
    props = pd.read_csv(clean_props_path)
    prefs = pd.read_csv(prefs_path)
    raw = load_raw_data()  # to get POIs
    print(f"✅ Loaded {len(props)} properties and {len(prefs)} preference rows")

    # User-based split to avoid leakage across groups
    user_ids = prefs.index.to_list()
    if MAX_PREFS is not None:
        user_ids = user_ids[:MAX_PREFS]
        print(f"⚠️  Limiting to first {len(user_ids)} users (MAX_PREFS={MAX_PREFS})")
    train_users, temp_users = train_test_split(user_ids, test_size=0.30, random_state=42)
    val_users, test_users = train_test_split(temp_users, test_size=0.50, random_state=42)

    print("🔧 Building training matches...")
    train_df, y_train, g_train = build_matches_for_users(props, prefs, raw.pois, train_users)
    print("🔧 Building validation matches...")
    val_df, y_val, g_val = build_matches_for_users(props, prefs, raw.pois, val_users)
    print("🔧 Building test matches...")
    test_df, y_test, g_test = build_matches_for_users(props, prefs, raw.pois, test_users)

    # Verify columns exist
    print(f"📊 Train DataFrame shape: {train_df.shape}, columns: {len(train_df.columns)}")
    if train_df.columns.duplicated().any():
        print("⚠️  Warning: Duplicate columns still present after deduplication")
        print(f"   Duplicate columns: {train_df.columns[train_df.columns.duplicated()].tolist()}")

    feature_cols = [
        "clean_price",
        "clean_area",
        "bedrooms",
        "bathrooms",
        "price_per_sqft",
        "condition_score",
        "luxury_indicator",
        "is_brand_new",
        "amenities_count",
        "crime_index",
        "crime_normalized",
        "safety_score",
        "risk_adjusted_safety",
        "risk_adjusted_roi",
        "price_fit_score",
        "bedroom_match",
        "area_match_score",
        "location_match_score",
        "facility_match_ratio",
        "school_score",
        "hospital_score",
        "metro_score",
        "park_score",
    ]

    # Verify all feature columns exist
    missing_cols = [col for col in feature_cols if col not in train_df.columns]
    if missing_cols:
        raise ValueError(f"Missing feature columns: {missing_cols}\n"
                        f"Available columns: {sorted(train_df.columns.tolist())}")
    
    print(f"✅ All {len(feature_cols)} feature columns found")

    pre = _build_preprocessor(feature_cols)
    
    # Configure GPU or CPU
    tree_method = "gpu_hist" if use_gpu else "hist"
    predictor = "gpu_predictor" if use_gpu else "auto"
    
    print(f"⚡ Using {'GPU' if use_gpu else 'CPU'} for training (tree_method={tree_method})")
    
    model = XGBRanker(
        tree_method=tree_method,
        predictor=predictor,
        learning_rate=0.1,
        n_estimators=300,
        max_depth=7,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="rank:pairwise",
        random_state=42,
        verbosity=2
    )
    pipe = Pipeline([("pre", pre), ("model", model)])
    print("⚡ Fitting XGBRanker...")
    pipe.fit(train_df, y_train, model__group=g_train)

    # Evaluate per split using mean NDCG across user groups
    print("📊 Evaluating model...")
    preds_train = pipe.predict(train_df)
    preds_val = pipe.predict(val_df)
    preds_test = pipe.predict(test_df)

    metrics = {
        "ndcg_train": mean_ndcg(y_train, preds_train, g_train),
        "ndcg_val": mean_ndcg(y_val, preds_val, g_val),
        "ndcg_test": mean_ndcg(y_test, preds_test, g_test),
        "train_rows": int(len(train_df)),
        "val_rows": int(len(val_df)),
        "test_rows": int(len(test_df)),
        "train_users": len(train_users),
        "val_users": len(val_users),
        "test_users": len(test_users),
    }

    model_path = os.path.join(ARTIFACT_DIR, "xgb_ranker.joblib")
    joblib.dump(pipe, model_path)
    print(f"💾 Model saved to: {model_path}")
    return RankerArtifacts(model_path=model_path, metrics=metrics)


print("✅ Training function ready")


In [ ]:
# Train the ranker
artifacts = train_ranker(
    clean_props_path="data/cleandataset/properties_clean.csv",
    prefs_path="data/cleandataset/preferences_enriched.csv",
    use_gpu=USE_GPU  # Use GPU if available
)

print("\n" + "="*60)
print("🎉 Training Complete!")
print("="*60)
print(f"\n📦 Model saved to: {artifacts.model_path}")
print(f"\n📊 Metrics:")
for key, value in artifacts.metrics.items():
    if isinstance(value, float):
        print(f"   {key}: {value:.4f}")
    else:
        print(f"   {key}: {value}")


## Download Model

Download the trained model and cleaned datasets.


In [ ]:
# Download model and data
from google.colab import files

print("📥 Downloading model...")
files.download("models_artifacts/xgb_ranker.joblib")

print("\n📥 Downloading cleaned datasets...")
files.download("data/cleandataset/properties_clean.csv")
files.download("data/cleandataset/preferences_enriched.csv")
